In [ ]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import os
import gc
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from detection_baselines.detection_vl_uncertainty import load_uncertainty_dataset, extract_uncertainty
from egh_vlm.utils import Qwen3ModelBundle, load_phd_dataset

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/vl_uncertainty_phd_qwen3.pt'

# Each sample requires N_PERTURBATIONS forward passes — budget ~5x the cost of stats extraction.
N_PERTURBATIONS = 5

#### Set Up

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH)

features = load_uncertainty_dataset(FEATURES_PATH) if os.path.isfile(FEATURES_PATH) else None
print(f'Length of processed features: {len(features) if features is not None else 0}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype=torch.float16,
    device_map=device
)
print('config.dtype:', model.config.dtype)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 1024)
model_bundle = Qwen3ModelBundle(model, processor, device)

#### Experiment

**Note (sentiment task):** Uncertainty scores for abstract `sentiment` questions will be noisy — the model may give inconsistent answers due to subjectivity rather than hallucination. Use per-task evaluation to isolate `object` vs `sentiment` performance.

In [ ]:
features = extract_uncertainty(
    dataset=dataset,
    model_bundle=model_bundle,
    client_type='qwen3',
    save_path=FEATURES_PATH,
    n_perturbations=N_PERTURBATIONS,
)
if features is not None and len(features) > 0:
    print('Sample uncertainty score:', features.scores[0].item())

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()